# 동물상 분류 모델 데이터
- 최소 데이터: 20개 클래스, 50장씩


## 1. 데이터 수집

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
import time

In [ ]:
from bing_image_downloader import downloader

downloader.download(
    query="알파카상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="공룡상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="원숭이상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="오리상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="코알라상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="늑대상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="햄스터상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="꽃돼지상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="너구리상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="말상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

## 2. 얼굴 추출
- 1) dataset/raw/<클래스>/(클래스당 70장)에서 이미지를 읽어 얼굴을 검출하고, 검출되면 랜드마크 기반으로 112×112 정렬 크롭(norm_crop)을 생성
    - 얼굴 검출/정렬이 실패한 이미지는 중앙 크롭 후 112×112로 리사이즈하여 대체
- 2) 검출 점수(det_score)와 정렬 성공 여부를 기반으로 점수를 매겨 클래스당 상위 50장을 dataset/selected50/<클래스>/에 저장

In [ ]:
import cv2, numpy as np, random
from pathlib import Path
from insightface.app import FaceAnalysis
from insightface.utils.face_align import norm_crop

# 유니코드 경로 읽기/쓰기 
def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def imwrite_unicode_jpg(path: Path, img):
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(".jpg", img)
    if not ok:
        raise RuntimeError(f"imencode failed: {path}")
    buf.tofile(str(path))

def center_square_resize(img, size=112):
    h, w = img.shape[:2]
    s = min(h, w)
    y1 = (h - s) // 2
    x1 = (w - s) // 2
    crop = img[y1:y1+s, x1:x1+s]
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# 설정
RAW_DIR = Path("dataset/raw")             # 클래스당 70장
SELECTED_DIR = Path("dataset/selected50") # 클래스당 50장 저장
TARGET_PER_CLASS = 50
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# 검출률 높이기
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(1280, 1280), det_thresh=0.1)

def detect_best_face(app, img):
    # 1) 원본
    faces = app.get(img)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img

    # 2) padding
    h, w = img.shape[:2]
    pad = int(0.25 * max(h, w))
    img_pad = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    faces = app.get(img_pad)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img_pad

    # 3) upsample
    scale = 1.5
    img_up = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    faces = app.get(img_up)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img_up

    return None, img

# raw(70) -> selected50(50)
SELECTED_DIR.mkdir(parents=True, exist_ok=True)

for cls_dir in [d for d in RAW_DIR.iterdir() if d.is_dir()]:
    imgs = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    if not imgs:
        print(f"[WARN] no images: {cls_dir.name}")
        continue

    random.shuffle(imgs)

    candidates = []  # (score, cropped112, src_path)
    for p in imgs:
        img = imread_unicode(p)
        if img is None:
            continue

        face, used_img = detect_best_face(app, img)

        if face is not None and getattr(face, "kps", None) is not None:
            out = norm_crop(used_img, face.kps)  # 112x112
            det_score = float(getattr(face, "det_score", 0.0))
            score = det_score + 1.0  # aligned 보너스
        else:
            out = center_square_resize(img, 112)
            score = 0.0  # fallback는 낮게

        candidates.append((score, out, p))

    if len(candidates) < TARGET_PER_CLASS:
        print(f"[WARN] {cls_dir.name}: candidates={len(candidates)} < target={TARGET_PER_CLASS} (그래도 진행)")

    candidates.sort(key=lambda x: x[0], reverse=True)
    selected = candidates[:TARGET_PER_CLASS]

    out_cls = SELECTED_DIR / cls_dir.name
    out_cls.mkdir(parents=True, exist_ok=True)

    for i, (score, out_img, src) in enumerate(selected, start=1):
        out_name = f"{cls_dir.name}_{i:03d}.jpg"
        imwrite_unicode_jpg(out_cls / out_name, out_img)

    print(f"selected50 {cls_dir.name}: selected={len(selected)} (top_score={selected[0][0] if selected else None})")

print("\nStep1 done:", SELECTED_DIR.resolve())

## 3. train / test 분리
- 1) dataset/selected50/<클래스>/의 이미지를 클래스별로 셔플한 뒤 80/20(train/test)로 분리
- 2) dataset/train/<클래스>/, dataset/test/<클래스>/에 복사

In [1]:
import random, shutil
from pathlib import Path

# 설정
SELECTED_DIR = Path("dataset/selected50")
TRAIN_DIR    = Path("dataset/train")
TEST_DIR     = Path("dataset/test")

TRAIN_RATIO = 0.8
SEED = 42
COPY_MODE = True         # True=복사, False=이동
CLEAR_OUTPUT = True      # train/test 폴더를 비우고 다시 생성

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

random.seed(SEED)

def copy_or_move(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if COPY_MODE:
        shutil.copy2(src, dst)   # 동일 파일명 있으면 덮어씀
    else:
        shutil.move(str(src), str(dst))

def clear_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def split_train_test(paths, train_ratio=0.8):
    paths = list(paths)
    random.shuffle(paths)
    n = len(paths)
    n_train = int(n * train_ratio)
    if n >= 2:
        n_train = min(max(n_train, 1), n - 1)  # 최소 1장씩은 배정
    return paths[:n_train], paths[n_train:]

# 실행
if not SELECTED_DIR.exists():
    raise FileNotFoundError(f"Not found: {SELECTED_DIR.resolve()}")

if CLEAR_OUTPUT:
    clear_dir(TRAIN_DIR)
    clear_dir(TEST_DIR)

for cls_dir in [d for d in SELECTED_DIR.iterdir() if d.is_dir()]:
    imgs = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    if not imgs:
        print(f"[WARN] no images: {cls_dir.name}")
        continue

    train_list, test_list = split_train_test(imgs, TRAIN_RATIO)

    for src in train_list:
        copy_or_move(src, TRAIN_DIR / cls_dir.name / src.name)

    for src in test_list:
        copy_or_move(src, TEST_DIR / cls_dir.name / src.name)

    print(f"split {cls_dir.name}: total={len(imgs)}, train={len(train_list)}, test={len(test_list)}")

print("\nDone!")
print("Train:", TRAIN_DIR.resolve())
print("Test :", TEST_DIR.resolve())

split 공룡상: total=50, train=40, test=10
split 꽃돼지상: total=50, train=40, test=10
split 너구리상: total=50, train=40, test=10
split 늑대상: total=50, train=40, test=10
split 말상: total=50, train=40, test=10
split 알파카상: total=50, train=40, test=10
split 오리상: total=50, train=40, test=10
split 원숭이상: total=50, train=40, test=10
split 코알라상: total=50, train=40, test=10
split 햄스터상: total=50, train=40, test=10

Done!
Train: C:\workspace\project2\gachikium\유현희\dataset\train
Test : C:\workspace\project2\gachikium\유현희\dataset\test


## 4. ArcFace 임베딩 → 분류기 학습/평가 코드

In [ ]:
import cv2, numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
TRAIN_DIR = Path("dataset/train")
TEST_DIR  = Path("dataset/test")

# 1) buffalo_l 로드 (필요시 자동으로 모델팩 다운로드 트리거)
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))   # GPU면 0, CPU면 -1

# 2) ArcFace 임베딩 모델 꺼내기 
rec = app.models.get("recognition")     # insightface의 recognition(ArcFace) 모델
if rec is None:
    raise RuntimeError(f"recognition model not found. keys={list(app.models.keys())}")

# 유니코드 경로 읽기
def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

# 임베딩 벡터를 L2 정규화(길이를 1로 맞춤)
# ArcFace 임베딩은 정규화 후 코사인 유사도 기반 비교가 일반적이고, 분류기(SVM/로지스틱) 학습 시에도 스케일이 안정적이라 성능이 좋아지는 경우가 많음.
def l2_normalize(x):
    return x / (np.linalg.norm(x) + 1e-12)

# 112x112 BGR 얼굴 이미지(정렬/크롭된 입력) -> 512차원 얼굴 임베딩 추출
def get_embedding_112_bgr(img_bgr_112):
    # 112x112 BGR 이미지 -> (512,) 임베딩
    if hasattr(rec, "get_feat"):
        emb = rec.get_feat(img_bgr_112)   # (1,512) 또는 (512,)
    else:
        raise RuntimeError(f"rec has no get_feat(). available: {dir(rec)}")

    emb = np.asarray(emb).reshape(-1)     # (512,)로 평탄화
    return l2_normalize(emb).astype(np.float32)

# train/test 폴더에서 각 이미지를 읽고 -> 임베딩(512-d)으로 변환 -> (X, y) 학습 데이터 형태로 만듦.
def load_split_embeddings(root_dir: Path):
    X, y = [], []
    classes = sorted([d.name for d in root_dir.iterdir() if d.is_dir()])
    class_to_idx = {c: i for i, c in enumerate(classes)}

    for cls in classes:
        cls_dir = root_dir / cls
        files = sorted(
            [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS],
            key=lambda p: p.name
        )

        for p in files:
            img = imread_unicode(p)
            if img is None:
                continue

            # 112x112 아니면 맞춰줌
            if img.shape[0] != 112 or img.shape[1] != 112:
                img = cv2.resize(img, (112, 112), interpolation=cv2.INTER_AREA)

            X.append(get_embedding_112_bgr(img))
            y.append(class_to_idx[cls])

    return np.vstack(X), np.array(y), classes

# 평가 함수
def evaluate(clf, X_test, y_test, classes):
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("test acc:", acc)

    print("\nclassification_report:")
    print(classification_report(y_test, pred, target_names=classes))

    print("\nconfusion_matrix:")
    print(confusion_matrix(y_test, pred))

# 데이터 준비
X_train, y_train, classes = load_split_embeddings(TRAIN_DIR)
X_test,  y_test,  _       = load_split_embeddings(TEST_DIR)

print("X_train:", X_train.shape, "X_test:", X_test.shape, "num_classes:", len(classes))

# 방법 1) Logistic Regression
def train_logistic(X_train, y_train):
    clf = LogisticRegression(max_iter=5000, n_jobs=-1)
    clf.fit(X_train, y_train)
    return clf

# 방법 2) Linear SVM
def train_linear_svm(X_train, y_train):
    clf = LinearSVC()
    clf.fit(X_train, y_train)
    return clf

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

## 5. 학습 / 예측

### 1) LogisticRegression

In [7]:
# 실행: 원하는 방법 선택
MODE = "logistic"   # "logistic" 또는 "svm"

if MODE == "logistic":
    clf = train_logistic(X_train, y_train)
elif MODE == "svm":
    clf = train_linear_svm(X_train, y_train)
else:
    raise ValueError('MODE must be "logistic" or "svm"')

evaluate(clf, X_test, y_test, classes)

test acc: 0.85

classification_report:
              precision    recall  f1-score   support

         공룡상       0.80      0.80      0.80        10
        꽃돼지상       1.00      0.70      0.82        10
        너구리상       0.90      0.90      0.90        10
         늑대상       0.77      1.00      0.87        10
          말상       1.00      0.80      0.89        10
        알파카상       0.75      0.90      0.82        10
         오리상       0.75      0.90      0.82        10
        원숭이상       0.82      0.90      0.86        10
        코알라상       1.00      0.60      0.75        10
        햄스터상       0.91      1.00      0.95        10

    accuracy                           0.85       100
   macro avg       0.87      0.85      0.85       100
weighted avg       0.87      0.85      0.85       100


confusion_matrix:
[[ 8  0  1  1  0  0  0  0  0  0]
 [ 0  7  0  0  0  0  2  0  0  1]
 [ 1  0  9  0  0  0  0  0  0  0]
 [ 0  0  0 10  0  0  0  0  0  0]
 [ 0  0  0  0  8  1  0  1  0  0]
 [ 0  0  0  1  0  

c:\workspace\project2\gachikium\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


꽃돼지상: recall 0.70 → 10장 중 3장을 다른 클래스로 보냄
-> 혼동행렬 보면 꽃돼지상(두 번째 행)에서 오리상으로 2장, 햄스터상으로 1장 이동.

코알라상: recall 0.60 → 10장 중 4장 틀림
-> 코알라상(아홉 번째 행)에서 늑대상/알파카상/오리상 쪽으로 흩어짐.

### 2) LinearSVC

In [8]:
# 실행: 원하는 방법 선택
MODE = "svm"   # "logistic" 또는 "svm"

if MODE == "logistic":
    clf = train_logistic(X_train, y_train)
elif MODE == "svm":
    clf = train_linear_svm(X_train, y_train)
else:
    raise ValueError('MODE must be "logistic" or "svm"')

evaluate(clf, X_test, y_test, classes)

test acc: 0.89

classification_report:
              precision    recall  f1-score   support

         공룡상       0.82      0.90      0.86        10
        꽃돼지상       1.00      1.00      1.00        10
        너구리상       1.00      0.90      0.95        10
         늑대상       0.83      1.00      0.91        10
          말상       1.00      0.90      0.95        10
        알파카상       0.82      0.90      0.86        10
         오리상       0.75      0.90      0.82        10
        원숭이상       0.89      0.80      0.84        10
        코알라상       1.00      0.70      0.82        10
        햄스터상       0.90      0.90      0.90        10

    accuracy                           0.89       100
   macro avg       0.90      0.89      0.89       100
weighted avg       0.90      0.89      0.89       100


confusion_matrix:
[[ 9  0  0  1  0  0  0  0  0  0]
 [ 0 10  0  0  0  0  0  0  0  0]
 [ 1  0  9  0  0  0  0  0  0  0]
 [ 0  0  0 10  0  0  0  0  0  0]
 [ 0  0  0  0  9  1  0  0  0  0]
 [ 0  0  0  0  0  

꽃돼지상: 완벽(10/10)

코알라상: 7/10 (여전히 약점이긴 함)

남는 헷갈림은 대체로 1장씩 정도로 소수.

## 6. 모델 저장

In [ ]:
import joblib
from pathlib import Path

MODEL_DIR = Path("model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {"clf": clf, "classes": classes},
    MODEL_DIR / "animal_face_clf.joblib"
)

print("saved:", (MODEL_DIR / "animal_face_clf.joblib").resolve())

## 7. 예측

In [10]:
import cv2, numpy as np, joblib
from pathlib import Path
from insightface.app import FaceAnalysis
from insightface.utils.face_align import norm_crop

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# 유니코드 경로 읽기
def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def l2_normalize(x):
    return x / (np.linalg.norm(x) + 1e-12)

# 1) 분류기 로드 (저장된 모델 로드)
bundle = joblib.load("model/animal_face_clf.joblib")
clf = bundle["clf"]
classes = bundle["classes"]

# 2) FaceAnalysis 준비 (det + recognition)
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(1280, 1280), det_thresh=0.1)

rec = app.models.get("recognition")
if rec is None:
    raise RuntimeError(f"recognition model not found. keys={list(app.models.keys())}")

def get_embedding_112_bgr(img_112):
    if hasattr(rec, "get_feat"):
        emb = rec.get_feat(img_112)
        emb = np.asarray(emb).reshape(-1)
    else:
        raise RuntimeError("rec.get_feat not found in recognition model.")
    return l2_normalize(emb).astype(np.float32)

def face_to_112(img_bgr):
    # 원본 BGR 이미지에서 가장 큰 얼굴을 찾아 112x112 정렬/크롭 반환. 실패 시 None.
    faces = app.get(img_bgr)
    if not faces:
        return None

    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))

    if getattr(face, "kps", None) is not None:
        return norm_crop(img_bgr, face.kps)  # 112x112
    else:
        x1, y1, x2, y2 = map(int, face.bbox)
        crop = img_bgr[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        if crop.size == 0:
            return None
        return cv2.resize(crop, (112, 112), interpolation=cv2.INTER_AREA)

def topk_candidates_from_clf(clf, emb_1x512, k=3):
    # 로지스틱이면 proba 기반, SVM이면 decision_function 기반으로 top-k 후보 추출
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(emb_1x512)[0]
        idxs = np.argsort(-proba)[:k]
        return [(classes[i], float(proba[i])) for i in idxs]

    if hasattr(clf, "decision_function"):
        scores = clf.decision_function(emb_1x512)
        scores = np.asarray(scores).reshape(-1)  # (num_classes,)
        idxs = np.argsort(-scores)[:k]
        # 점수는 확률이 아니라 상대적인 점수임
        return [(classes[i], float(scores[i])) for i in idxs]

    return None

def predict_one(image_path: str, topk=3):
    # 이미지 1장 예측
    p = Path(image_path)
    img = imread_unicode(p)
    if img is None:
        return {"path": str(p), "ok": False, "reason": "failed_to_read"}

    img112 = face_to_112(img)
    if img112 is None:
        return {"path": str(p), "ok": False, "reason": "no_face_detected"}

    emb = get_embedding_112_bgr(img112)
    emb_1x = emb.reshape(1, -1)

    pred_idx = int(clf.predict(emb_1x)[0])
    pred_name = classes[pred_idx]

    topk_list = topk_candidates_from_clf(clf, emb_1x, k=topk)

    return {
        "path": str(p),
        "ok": True,
        "pred": pred_name,
        "topk": topk_list,
    }

def predict_path(path: str, topk=3, recursive=True):
    # path가 파일이면 1장 예측, 폴더면 폴더 내 이미지 전부 예측. recursive=True면 하위 폴더까지 전부 찾음.
    p = Path(path)
    if p.is_file():
        return [predict_one(str(p), topk=topk)]

    if p.is_dir():
        it = p.rglob("*") if recursive else p.glob("*")
        files = [x for x in it if x.is_file() and x.suffix.lower() in IMG_EXTS]
        files.sort(key=lambda x: str(x))
        return [predict_one(str(f), topk=topk) for f in files]

    raise FileNotFoundError(f"not found: {p}")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

### 1) 예측할 파일 1개

In [12]:
results = predict_path("./infer/testImg1.png")
print(results[:3])

c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


[{'path': 'infer\\testImg1.png', 'ok': True, 'pred': '원숭이상', 'topk': [('원숭이상', -0.3434114672361223), ('너구리상', -0.5446494394000524), ('말상', -0.7213288731460767)]}]


### 2) 예측할 폴더

In [13]:
results = predict_path("./infer")
print(results[:3])

c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


[{'path': 'infer\\testImg1.png', 'ok': True, 'pred': '원숭이상', 'topk': [('원숭이상', -0.3434114672361223), ('너구리상', -0.5446494394000524), ('말상', -0.7213288731460767)]}, {'path': 'infer\\testImg2.png', 'ok': True, 'pred': '원숭이상', 'topk': [('원숭이상', -0.1422636506721755), ('오리상', -0.31935528721422446), ('코알라상', -0.526302416484229)]}]


c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


#### * 보기 좋게 출력

In [14]:
def pretty_print_results(results, topk=3):
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] {r['path']}")
        if not r.get("ok", False):
            print(f"FAIL: {r.get('reason')}")
            continue

        print(f"PRED: {r['pred']}")
        tk = r.get("topk")
        if tk:
            print("TOP{}:".format(min(topk, len(tk))))
            for rank, (name, score) in enumerate(tk[:topk], 1):
                # SVM decision score는 확률이 아니라 점수라서 “score”로 표시
                print(f"    {rank}. {name:>6}   score={score:+.3f}")

pretty_print_results(results, topk=3)


[1] infer\testImg1.png
PRED: 원숭이상
TOP3:
    1.   원숭이상   score=-0.343
    2.   너구리상   score=-0.545
    3.     말상   score=-0.721

[2] infer\testImg2.png
PRED: 원숭이상
TOP3:
    1.   원숭이상   score=-0.142
    2.    오리상   score=-0.319
    3.   코알라상   score=-0.526
